# OrigenLM V2 · Kaggle T4 (reanudable)

Entrena **solo pesos propios** de OrigenLM V2 (6.819.072 parámetros) reutilizando
el checkpoint, el estado del optimizador, el tokenizador y el corpus ya preparados.
No descarga ni importa pesos de DeepSeek, Qwen, Llama u otros modelos.

**Antes de ejecutar necesitas tres cosas, y las subes tú:**
1. Dataset privado con el corpus: `train.bin`, `val.bin`, `tokenizer.json`, `meta.json`, `hashes.json`.
2. Dataset privado con el checkpoint: `origen_v2_es.pt`.
3. *Internet* activado en el notebook (para clonar el repo e instalar `tokenizers`).

Los datasets de Kaggle son privados por defecto, pero **siguen siendo una subida a un
servicio externo**: no lo hagas sin autorización explícita del dueño del corpus.

In [ ]:
import subprocess, sys, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv'], capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0), '| bf16', torch.cuda.is_bf16_supported())
else:
    print('SIN GPU: activa Accelerator > GPU T4 x2 en Settings y reinicia el notebook')

In [ ]:
REPO = 'https://github.com/J3lk5d0x3l/OrigenLM'
COMMIT = 'CAMBIAR_POR_EL_SHA_DEL_PR'  # fija el commit para que el run sea reproducible
BASE = '/kaggle/working'

get_ipython().system(f'rm -rf {BASE}/repo && git clone --quiet {REPO} {BASE}/repo')
get_ipython().system(f'cd {BASE}/repo && git checkout --quiet {COMMIT} && git log --oneline -1')
get_ipython().system(f'{sys.executable} -m pip install -q "tokenizers>=0.20,<1"')
sys.path.insert(0, f'{BASE}/repo')
print('repo listo en', f'{BASE}/repo')

## Datos privados

Sube (cuando lo autorices) dos datasets privados desde tu equipo:

```bash
# en el Codespace: exporta y calcula hashes sin publicar nada
python -m scripts.exportar_checkpoint --salida dist
```

Luego, en tu PC (requiere `pip install kaggle` y tu `kaggle.json`):

```bash
kaggle datasets create -p dist/corpus --dir-mode zip      # privado por defecto
kaggle datasets create -p dist/checkpoint --dir-mode zip
```

Ajusta los slugs de la celda siguiente a los que te devuelva Kaggle
(`tu-usuario/origenlm-v2-es` y `tu-usuario/origenlm-v2-checkpoint`).

Los hashes se verifican antes de entrenar: si el corpus y el checkpoint no son
los mismos que en el Codespace, el notebook se detiene en vez de entrenar basura.

In [ ]:
from pathlib import Path
import hashlib, json, shutil

CORPUS_IN = Path('/kaggle/input/origenlm-v2-es')            # <-- tu slug
CKPT_IN = Path('/kaggle/input/origenlm-v2-checkpoint')      # <-- tu slug
DATOS = Path('/kaggle/working/datos')
CKPT = Path('/kaggle/working/origen_v2_es.pt')

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for bloque in iter(lambda: f.read(1 << 20), b''):
            h.update(bloque)
    return h.hexdigest()

DATOS.mkdir(parents=True, exist_ok=True)
for nombre in ('train.bin', 'val.bin', 'tokenizer.json', 'meta.json'):
    shutil.copy2(CORPUS_IN / nombre, DATOS / nombre)
if not CKPT.exists():
    shutil.copy2(CKPT_IN / 'origen_v2_es.pt', CKPT)

esperados = json.loads((CORPUS_IN / 'hashes.json').read_text())
fallos = []
rutas = {**{n: DATOS / n for n in ('train.bin', 'val.bin', 'tokenizer.json')},
         'checkpoint': CKPT}
for clave, ruta in rutas.items():
    if clave not in esperados:
        continue
    real = sha256(ruta)
    print(f'{clave:12s} {real[:16]}…', 'OK' if real == esperados[clave] else 'NO COINCIDE')
    if real != esperados[clave]:
        fallos.append(clave)
assert not fallos, f'Archivos alterados: {fallos}'

In [ ]:
# Benchmark real: 30 pasos, sin tocar el checkpoint de producción.
get_ipython().system(
    f'cd {BASE}/repo && {sys.executable} train_v2_bpe.py '
    f'--datos {DATOS} --salida {BASE}/bench.pt --overwrite '
    f'--pasos 30 --ctx 512 --batch-size 4 --grad-accum 4 --warmup 5 '
    f'--eval-cada 1000 --guardar-cada 1000 --device cuda --precision auto')

## Entrenamiento

`--pasos-total 12000` fija el horizonte del coseno en ~98M tokens (≈0,9 épocas de
los 112M de entrenamiento). `--max-minutos 55` deja margen para guardar antes del
límite de 9 h de la sesión de Kaggle. Cada 100 pasos se escribe un checkpoint
atómico, y `SIGTERM`/`SIGINT` (parada de Kaggle, botón de stop) guarda el paso en
curso antes de salir. Para seguir en otra sesión, vuelve a ejecutar la misma celda:
reanuda desde `/kaggle/working/origen_v2_es.pt` sin reiniciar el warmup.

In [ ]:
get_ipython().system(
    f'cd {BASE}/repo && {sys.executable} train_v2_bpe.py '
    f'--datos {DATOS} --salida {CKPT} --resume {CKPT} '
    f'--pasos 20000 --pasos-total 12000 --ctx 512 --batch-size 4 --grad-accum 4 '
    f'--lr 6e-4 --warmup 200 --eval-cada 250 --guardar-cada 100 '
    f'--max-minutos 55 --device cuda --precision auto')

In [ ]:
# Evaluación honesta: NLL/perplejidad en validación + 20 pruebas escritas a mano.
get_ipython().system(
    f'cd {BASE}/repo && {sys.executable} -m scripts.evaluar_modelo '
    f'--checkpoint {CKPT} --tokenizer {DATOS}/tokenizer.json --datos {DATOS} '
    f'--salida {BASE}/informe.json --device cuda')

import torch
carga = torch.load(CKPT, map_location='cpu', weights_only=True)
print('paso', carga['step'], '| total', carga['total_steps'],
      '| val', carga['val_loss'], '| tokens', carga['tokens_vistos'])
print('Descarga /kaggle/working/origen_v2_es.pt y /kaggle/working/informe.json',
      'desde la pestaña Output antes de cerrar la sesión.')

## Siguiente sesión

1. Guarda `origen_v2_es.pt` e `informe.json` (pestaña *Output*).
2. Si quieres seguir, sube ese checkpoint como nueva versión del dataset privado.
3. Reejecuta las celdas: el `--resume` continúa paso, optimizador y curva de LR.

Límites de Kaggle a tener en cuenta: 30 h de GPU por semana y ~20 min de
inactividad antes de parar el kernel (el guardado por señal cubre ese caso).
El notebook no publica nada: solo escribe en `/kaggle/working`.